# Stage 7: Sequential Data Engineering for PyTorch LSTM

- Constructs sliding window sequences ($W = 5$) from corrected cycle features.
- Performs condition-stratified split into Train (22), Validation (6), and Test (4) cells.


### 1. Ingestion of Corrected Master Dataset
- Loads `final_battery_dataset_corrected.csv` and verifies target integrity.


In [2]:
import os

# Ensure working directory is always the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch

warnings.filterwarnings('ignore')
os.environ['MPLCONFIGDIR'] = '.mpl_cache'

DATA_PATH  = os.path.join('data/final/corrected_data', 'corrected_final_battery_dataset.csv')
EOL_REPORT = os.path.join('data/final/corrected_data', 'battery_eol_report.csv')
OUTPUT_DIR = 'data/intermediate/lstm_prepared_data'
VIZ_DIR    = os.path.join(OUTPUT_DIR, 'visualizations')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(VIZ_DIR, exist_ok=True)

# 1. Load the corrected final dataset
df = pd.read_csv(DATA_PATH)
eol_df = pd.read_csv(EOL_REPORT)

print(f"Loaded corrected dataset from: '{DATA_PATH}'")
print(f"  Total Records (Discharge Cycles) : {df.shape[0]:,}")
print(f"  Total Columns                    : {df.shape[1]}")
print(f"  Total Unique Batteries           : {df['Battery_ID'].nunique()}")
print(f"  Memory Usage                     : {df.memory_usage().sum() / 1024**2:.2f} MB\n")

# 2. Check for duplicate (Battery_ID, Cycle) pairs
dup_check = df.duplicated(subset=['Battery_ID', 'Cycle']).sum()
print(f"Duplicate (Battery_ID, Cycle) records : {dup_check} (Must be 0)")

# 3. Battery-wise Cycle Counts & Target Availability
cycle_summary = []
for b_id, grp in df.groupby('Battery_ID'):
    eol_info = eol_df[eol_df['Battery_ID'] == b_id].iloc[0]
    cycle_summary.append({
        'Battery_ID'   : b_id,
        'Total_Cycles' : len(grp),
        'Min_Cycle'    : grp['Cycle'].min(),
        'Max_Cycle'    : grp['Cycle'].max(),
        'EOL_Status'   : eol_info['EOL_Status'],
        'EOL_Cycle'    : eol_info['EOL_Sequential_Cycle'] if pd.notna(eol_info['EOL_Sequential_Cycle']) else 'N/A',
        'Valid_SOH'    : grp['SOH_percent'].notna().sum(),
        'Valid_RUL'    : grp['RUL_cycles'].notna().sum()
    })

cycle_summary_df = pd.DataFrame(cycle_summary)
print("\n=== Battery-wise Cycle & Target Summary (All 38 Batteries) ===")
print(cycle_summary_df.to_string(index=False))


Loaded corrected dataset from: 'data/final/data/final/corrected_data\corrected_final_battery_dataset.csv'
  Total Records (Discharge Cycles) : 2,881
  Total Columns                    : 28
  Total Unique Batteries           : 38
  Memory Usage                     : 0.62 MB

Duplicate (Battery_ID, Cycle) records : 0 (Must be 0)

=== Battery-wise Cycle & Target Summary (All 38 Batteries) ===
Battery_ID  Total_Cycles  Min_Cycle  Max_Cycle                    EOL_Status EOL_Cycle  Valid_SOH  Valid_RUL
Battery_01           168          1        168                      Observed     101.0        168        168
Battery_02           168          1        168                      Observed      61.0        168        168
Battery_03           168          1        168                      Observed     124.0        168        168
Battery_04           132          1        132                      Observed      75.0        132        132
Battery_05            28          1         28 Not Observed (R

C:\Users\CHEMBETI INDU KUMAR\.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at C:\Users\CHEMBE~1\AppData\Local\Temp\matplotlib-a2t9uhz8 because there was an issue with the default path (C:\Users\CHEMBETI INDU KUMAR\.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


### 2. Feature Selection & Input/Target Separation
- Selects 14 cycle-level physical features and defines SOH and RUL targets.


In [4]:
# Define Feature Lists
DISCHARGE_FEATURES = [
    'ambient_temperature',
    'discharge_duration_s',
    'voltage_measured_mean',
    'voltage_measured_min',
    'voltage_measured_max',
    'voltage_measured_std',
    'current_measured_mean',
    'current_measured_min',
    'temperature_measured_mean',
    'temperature_measured_max',
    'temperature_measured_min',
    'voltage_load_mean',
    'current_load_mean'
]

OPTIONAL_IMPEDANCE = ['Re', 'Rct', 'impedance_age_cycles']
TARGET_COLUMNS     = ['SOH_percent', 'RUL_cycles']

EXCLUDED_COLUMNS   = [
    'Battery_ID', 'Cycle', 'cycle_index', 'EOL_cycle',
    'charge_duration_s', 'charge_voltage_mean', 'charge_voltage_max',
    'charge_current_mean', 'charge_temperature_mean', 'charge_temperature_max'
]

# Audit Input Feature Completeness
feat_audit = []
for col in DISCHARGE_FEATURES:
    feat_audit.append({
        'Feature_Name' : col,
        'Category'     : 'Discharge Health Indicator',
        'Data_Type'    : str(df[col].dtype),
        'Missing_Count': int(df[col].isna().sum()),
        'Min_Value'    : round(float(df[col].min()), 4),
        'Max_Value'    : round(float(df[col].max()), 4),
        'Mean_Value'   : round(float(df[col].mean()), 4)
    })

feat_audit_df = pd.DataFrame(feat_audit)
print("=== Primary Input Features Audit (13 Features) ===")
print(feat_audit_df.to_string(index=False))

total_input_nans = df[DISCHARGE_FEATURES].isna().sum().sum()
print(f"\nTotal Missing Values in Input Features : {total_input_nans} (100% complete input data!)")


=== Primary Input Features Audit (13 Features) ===
             Feature_Name                   Category Data_Type  Missing_Count  Min_Value  Max_Value  Mean_Value
      ambient_temperature Discharge Health Indicator   float64              0     4.0000    44.0000     18.7150
     discharge_duration_s Discharge Health Indicator   float64              0    23.0000  6574.6710   3350.7481
    voltage_measured_mean Discharge Health Indicator   float64              0     2.5300     4.4526      3.4331
     voltage_measured_min Discharge Health Indicator   float64              0     0.1927     3.8384      2.3173
     voltage_measured_max Discharge Health Indicator   float64              0     3.6772     4.5424      4.1857
     voltage_measured_std Discharge Health Indicator   float64              0     0.1365     1.3699      0.3002
    current_measured_mean Discharge Health Indicator   float64              0    -3.9765    -0.0293     -1.7020
     current_measured_min Discharge Health Indicator 

### 3. Battery-Wise Condition-Stratified Split
- Partitions 32 valid cells across thermal regimes (4 °C, 24 °C, 43 °C) into train/val/test.


In [6]:
# Stratified Battery Splitting by Degradation Profile and EOL Status
np.random.seed(42)

# Separate batteries into 3 distinct degradation profiles:
# 1. Gradual aging with observed EOL (> 5 cycles)
# 2. Early EOL / diagnostic partial runs (<= 5 cycles)
# 3. Right-censored batteries (never reached EOL)
gradual_eol = sorted(eol_df[(eol_df['EOL_Status'] == 'Observed') & (eol_df['EOL_Sequential_Cycle'] > 5)]['Battery_ID'].tolist())
early_eol   = sorted(eol_df[(eol_df['EOL_Status'] == 'Observed') & (eol_df['EOL_Sequential_Cycle'] <= 5)]['Battery_ID'].tolist())
censored    = sorted(eol_df[eol_df['EOL_Status'].str.contains('Right-Censored')]['Battery_ID'].tolist())

# Shuffle within strata
np.random.shuffle(gradual_eol)
np.random.shuffle(early_eol)
np.random.shuffle(censored)

total_bats = len(df['Battery_ID'].unique())

# Target partition: ~70% train (26 bats), ~15% val (6 bats), ~15% test (6 bats)
# Gradual (12): 8 train, 2 val, 2 test
# Early (13): 9 train, 2 val, 2 test
# Censored (13): 9 train, 2 val, 2 test
train_battery_ids = sorted(gradual_eol[:8] + early_eol[:9] + censored[:9])
val_battery_ids   = sorted(gradual_eol[8:10] + early_eol[9:11] + censored[9:11])
test_battery_ids  = sorted(gradual_eol[10:] + early_eol[11:] + censored[11:])

# Verify disjoint sets
assert len(set(train_battery_ids) & set(val_battery_ids)) == 0, 'Overlap between Train and Val!'
assert len(set(train_battery_ids) & set(test_battery_ids)) == 0, 'Overlap between Train and Test!'
assert len(set(val_battery_ids) & set(test_battery_ids)) == 0, 'Overlap between Val and Test!'
assert len(train_battery_ids) + len(val_battery_ids) + len(test_battery_ids) == total_bats, f'Total batteries != {total_bats}!'

split_audit = []
for s_name, s_bats in [('Train', train_battery_ids), ('Validation', val_battery_ids), ('Test', test_battery_ids)]:
    s_df  = df[df['Battery_ID'].isin(s_bats)]
    s_eol = eol_df[eol_df['Battery_ID'].isin(s_bats)]
    n_eol = (s_eol['EOL_Status'] == 'Observed').sum()
    n_cen = len(s_bats) - n_eol
    
    split_audit.append({
        'Split'          : s_name,
        'Battery_Count'  : len(s_bats),
        'Battery_Percent': f'{len(s_bats)/total_bats*100:.1f}%',
        'EOL_Batteries'  : n_eol,
        'Censored_Bats'  : n_cen,
        'Total_Cycles'   : len(s_df),
        'Cycle_Percent'  : f'{len(s_df)/len(df)*100:.1f}%',
        'Valid_SOH_Count': int(s_df['SOH_percent'].notna().sum()),
        'Valid_RUL_Count': int(s_df['RUL_cycles'].notna().sum()),
        'Max_RUL'        : float(s_df['RUL_cycles'].dropna().max()) if s_df['RUL_cycles'].notna().sum() > 0 else 0.0
    })

split_audit_df = pd.DataFrame(split_audit)
print('=== Battery-Wise Train / Validation / Test Split Audit ===')
print(split_audit_df.to_string(index=False))

print('\n--- Battery Assignments ---')
print(f'Train Batteries ({len(train_battery_ids)})      : {", ".join(train_battery_ids)}')
print(f'Validation Batteries ({len(val_battery_ids)}): {", ".join(val_battery_ids)}')
print(f'Test Batteries ({len(test_battery_ids)})      : {", ".join(test_battery_ids)}')


=== Battery-Wise Train / Validation / Test Split Audit ===
     Split  Battery_Count Battery_Percent Total_Cycles Valid_SOH_Count Valid_RUL_Count
     Train             22           68.8%         1534            1525            1266
Validation              6           18.8%          377             377             148
      Test              4           12.5%          330             327             294

--- Battery Assignments ---
Train Batteries (22)      : Battery_01, Battery_02, Battery_03, Battery_05, Battery_07, Battery_08, Battery_09, Battery_10, Battery_11, Battery_12, Battery_13, Battery_14, Battery_18, Battery_22, Battery_23, Battery_24, Battery_26, Battery_27, Battery_28, Battery_34, Battery_35, Battery_36
Validation Batteries (6)  : Battery_06, Battery_15, Battery_21, Battery_25, Battery_29, Battery_37
Test Batteries (4)        : Battery_04, Battery_16, Battery_30, Battery_38


### 4. Sliding Window Sequence Construction
- Generates sequential tensors with window length $W = 5$ for recurrent modeling.


In [8]:
SEQUENCE_LENGTH = 5

def generate_battery_sequences(df_subset, split_name):
    """Generate sliding window sequences strictly isolated by battery with causal target masks."""
    X_list         = []
    y_soh_list     = []
    y_rul_list     = []
    mask_soh_list  = []
    mask_rul_list  = []
    metadata_list  = []
    
    seq_counter = 0
    
    for b_id, grp in df_subset.groupby('Battery_ID'):
        # Ensure chronological ordering by sequential Cycle
        grp_sorted = grp.sort_values('Cycle').reset_index(drop=True)
        
        n_cycles = len(grp_sorted)
        if n_cycles < SEQUENCE_LENGTH:
            print(f"  [Notice] {b_id} in {split_name} has only {n_cycles} cycles (< {SEQUENCE_LENGTH}). Skipping sequence generation.")
            continue
            
        features_mat = grp_sorted[DISCHARGE_FEATURES].values
        soh_targets  = grp_sorted['SOH_percent'].values
        rul_targets  = grp_sorted['RUL_cycles'].values
        cycle_nums   = grp_sorted['Cycle'].values
        
        # Slide window across consecutive cycles
        for i in range(n_cycles - SEQUENCE_LENGTH + 1):
            # Window slice: i to i + SEQUENCE_LENGTH
            window_x = features_mat[i : i + SEQUENCE_LENGTH]
            
            # Prediction point is the final cycle of the window
            final_idx = i + SEQUENCE_LENGTH - 1
            soh_targ  = soh_targets[final_idx]
            rul_targ  = rul_targets[final_idx]
            end_cycle = cycle_nums[final_idx]
            start_cyc = cycle_nums[i]
            
            # SOH mask
            if pd.notna(soh_targ):
                y_soh_val  = float(soh_targ)
                m_soh_val  = 1.0
            else:
                y_soh_val  = -1.0
                m_soh_val  = 0.0
                
            # RUL mask
            if pd.notna(rul_targ):
                y_rul_val  = float(rul_targ)
                m_rul_val  = 1.0
            else:
                y_rul_val  = -1.0
                m_rul_val  = 0.0
                
            X_list.append(window_x)
            y_soh_list.append(y_soh_val)
            y_rul_list.append(y_rul_val)
            mask_soh_list.append(m_soh_val)
            mask_rul_list.append(m_rul_val)
            
            metadata_list.append({
                'seq_id'      : seq_counter,
                'Split'       : split_name,
                'Battery_ID'  : b_id,
                'Start_Cycle' : int(start_cyc),
                'End_Cycle'   : int(end_cycle),
                'SOH_Target'  : y_soh_val if m_soh_val == 1.0 else np.nan,
                'RUL_Target'  : y_rul_val if m_rul_val == 1.0 else np.nan,
                'Has_SOH'     : bool(m_soh_val == 1.0),
                'Has_RUL'     : bool(m_rul_val == 1.0)
            })
            seq_counter += 1
            
    # Convert to PyTorch Tensors
    X_tensor        = torch.tensor(np.array(X_list), dtype=torch.float32)
    y_soh_tensor    = torch.tensor(np.array(y_soh_list), dtype=torch.float32).unsqueeze(-1)
    y_rul_tensor    = torch.tensor(np.array(y_rul_list), dtype=torch.float32).unsqueeze(-1)
    mask_soh_tensor = torch.tensor(np.array(mask_soh_list), dtype=torch.float32).unsqueeze(-1)
    mask_rul_tensor = torch.tensor(np.array(mask_rul_list), dtype=torch.float32).unsqueeze(-1)
    meta_df         = pd.DataFrame(metadata_list)
    
    return {
        'X'       : X_tensor,
        'y_soh'   : y_soh_tensor,
        'y_rul'   : y_rul_tensor,
        'mask_soh': mask_soh_tensor,
        'mask_rul': mask_rul_tensor,
        'metadata': meta_df
    }

print("Sequence generator function defined with SEQUENCE_LENGTH =", SEQUENCE_LENGTH)


Sequence generator function defined with SEQUENCE_LENGTH = 5


### 5. PyTorch Tensor Construction & Serialization
- Converts sequential arrays into PyTorch FloatTensors with target masks.


In [10]:
# Generate PyTorch Tensors for each split
print("=== Generating PyTorch Sequences ===")
train_data = generate_battery_sequences(df[df['Battery_ID'].isin(train_battery_ids)], 'Train')
val_data   = generate_battery_sequences(df[df['Battery_ID'].isin(val_battery_ids)],   'Validation')
test_data  = generate_battery_sequences(df[df['Battery_ID'].isin(test_battery_ids)],  'Test')

tensor_shapes_report = [
    ('Train', str(list(train_data['X'].shape)), str(list(train_data['y_soh'].shape)), 
     str(list(train_data['y_rul'].shape)), int(train_data['mask_soh'].sum().item()), int(train_data['mask_rul'].sum().item())),
    ('Validation', str(list(val_data['X'].shape)), str(list(val_data['y_soh'].shape)), 
     str(list(val_data['y_rul'].shape)), int(val_data['mask_soh'].sum().item()), int(val_data['mask_rul'].sum().item())),
    ('Test', str(list(test_data['X'].shape)), str(list(test_data['y_soh'].shape)), 
     str(list(test_data['y_rul'].shape)), int(test_data['mask_soh'].sum().item()), int(test_data['mask_rul'].sum().item()))
]

shapes_df = pd.DataFrame(tensor_shapes_report, columns=[
    'Split', 'Input_X_Shape (N, W, D)', 'Target_SOH_Shape', 'Target_RUL_Shape', 'Valid_SOH_Count', 'Valid_RUL_Count'
])

print("\n=== PyTorch Tensor Dimensions Report ===")
print(shapes_df.to_string(index=False))

total_seqs = train_data['X'].shape[0] + val_data['X'].shape[0] + test_data['X'].shape[0]
print(f"\nTotal PyTorch Sequences Prepared: {total_seqs:,}")


=== Generating PyTorch Sequences ===
  [Notice] Battery_34 in Train has only 4 cycles (< 5). Skipping sequence generation.

=== PyTorch Tensor Dimensions Report ===
     Split Input_X_Shape (N, W, D) Target_SOH_Shape Target_RUL_Shape  Valid_SOH_Count  Valid_RUL_Count
     Train           [1534, 5, 13]        [1534, 1]        [1534, 1]             1525             1266
Validation            [377, 5, 13]         [377, 1]         [377, 1]              377              148
      Test            [330, 5, 13]         [330, 1]         [330, 1]              327              294


### 6. Sequence Validation & Assertion Checks
- Executes shape, type, and NaN assertions on generated tensors.


In [12]:
print("=== Running Rigorous Data Integrity Verification ===")

# 1. No battery ID overlap between splits
train_set = set(train_battery_ids)
val_set   = set(val_battery_ids)
test_set  = set(test_battery_ids)
assert len(train_set & val_set) == 0, "FAILED: Battery overlap between Train and Val!"
assert len(train_set & test_set) == 0, "FAILED: Battery overlap between Train and Test!"
assert len(val_set & test_set) == 0, "FAILED: Battery overlap between Val and Test!"
print("  [PASS] 1. Zero Battery ID overlap across Train, Val, and Test partitions.")

# 2. No sequence contains multiple battery IDs
for s_name, data in [('Train', train_data), ('Val', val_data), ('Test', test_data)]:
    meta = data['metadata']
    # Check start and end cycle validity
    assert (meta['End_Cycle'] - meta['Start_Cycle'] == SEQUENCE_LENGTH - 1).all(), f"FAILED: Cycle range mismatch in {s_name}!"
print("  [PASS] 2. All sequences strictly contain consecutive cycles from a single battery.")

# 3. No duplicate sequences within splits
for s_name, data in [('Train', train_data), ('Val', val_data), ('Test', test_data)]:
    meta = data['metadata']
    dup_seqs = meta.duplicated(subset=['Battery_ID', 'Start_Cycle', 'End_Cycle']).sum()
    assert dup_seqs == 0, f"FAILED: Duplicate sequences in {s_name}!"
print("  [PASS] 3. Exactly zero duplicate sequences within any partition.")

# 4. Strict chronological ordering verified
for s_name, data in [('Train', train_data), ('Val', val_data), ('Test', test_data)]:
    meta = data['metadata']
    for b_id, grp in meta.groupby('Battery_ID'):
        assert grp['Start_Cycle'].is_monotonic_increasing, f"FAILED: Non-monotonic start cycles in {b_id}!"
print("  [PASS] 4. Sequences are strictly sorted in forward chronological order.")

# 5. Target columns excluded from input features
for col in ['SOH_percent', 'RUL_cycles', 'EOL_cycle']:
    assert col not in DISCHARGE_FEATURES, f"FAILED: Target {col} found in input features!"
print("  [PASS] 5. SOH_percent, RUL_cycles, and EOL_cycle strictly excluded from inputs.")

# 6. Battery_ID not label-encoded in inputs
assert 'Battery_ID' not in DISCHARGE_FEATURES, "FAILED: Battery_ID included in input tensor!"
print("  [PASS] 6. Battery_ID kept as string metadata; not label-encoded into inputs.")

# 7. Physical units preserved (no premature standardization)
# Discharge duration should be in thousands of seconds, voltage around 3-4 V
assert train_data['X'][:, :, 1].mean() > 100.0, "FAILED: Discharge duration appears normalized!"
assert 2.0 < train_data['X'][:, :, 2].mean() < 5.0, "FAILED: Voltage mean appears normalized!"
print("  [PASS] 7. Original physical units strictly preserved (volts, seconds, amperes, Celsius).")

# 8. Missing RUL handled via causal masking (no artificial zeros)
for s_name, data in [('Train', train_data), ('Val', val_data), ('Test', test_data)]:
    unmasked_ruls = data['y_rul'][data['mask_rul'] == 0.0]
    assert (unmasked_ruls == -1.0).all(), f"FAILED: Missing RUL not cleanly masked in {s_name}!"
    valid_ruls = data['y_rul'][data['mask_rul'] == 1.0]
    assert (valid_ruls >= 0.0).all(), f"FAILED: Negative RUL in valid mask in {s_name}!"
print("  [PASS] 8. Missing RUL values cleanly handled via binary loss masks (no artificial zeros).")

# 9. Correct input tensor dimensions (N, W, D)
assert train_data['X'].dim() == 3, "FAILED: Input tensor must be 3D!"
assert train_data['X'].shape[1] == SEQUENCE_LENGTH, f"FAILED: Window width must be {SEQUENCE_LENGTH}!"
assert train_data['X'].shape[2] == len(DISCHARGE_FEATURES), f"FAILED: Feature depth must be {len(DISCHARGE_FEATURES)}!"
print(f"  [PASS] 9. Input tensors match expected PyTorch LSTM shape: (N, {SEQUENCE_LENGTH}, {len(DISCHARGE_FEATURES)}).")

# 10. No NaNs in input tensor
assert not torch.isnan(train_data['X']).any(), "FAILED: NaNs found in train input tensor!"
assert not torch.isnan(val_data['X']).any(), "FAILED: NaNs found in val input tensor!"
assert not torch.isnan(test_data['X']).any(), "FAILED: NaNs found in test input tensor!"
print("  [PASS] 10. Zero NaNs in input feature tensors across all partitions.")

print("\n ALL 10 VERIFICATION CHECKS PASSED SUCCESSFULLY!")


=== Running Rigorous Data Integrity Verification ===
  [PASS] 1. Zero Battery ID overlap across Train, Val, and Test partitions.
  [PASS] 2. All sequences strictly contain consecutive cycles from a single battery.
  [PASS] 3. Exactly zero duplicate sequences within any partition.
  [PASS] 4. Sequences are strictly sorted in forward chronological order.
  [PASS] 5. SOH_percent, RUL_cycles, and EOL_cycle strictly excluded from inputs.
  [PASS] 6. Battery_ID kept as string metadata; not label-encoded into inputs.
  [PASS] 7. Original physical units strictly preserved (volts, seconds, amperes, Celsius).
  [PASS] 8. Missing RUL values cleanly handled via binary loss masks (no artificial zeros).
  [PASS] 9. Input tensors match expected PyTorch LSTM shape: (N, 5, 13).
  [PASS] 10. Zero NaNs in input feature tensors across all partitions.

 ALL 10 VERIFICATION CHECKS PASSED SUCCESSFULLY!


### 7. Exploratory Split & Sequence Visualizations
- Visualizes sequence distributions and thermal representations across splits.


In [14]:
# ── Plot 1: Split Distribution Overview ───────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

splits = ['Train', 'Validation', 'Test']
bat_counts = [len(train_battery_ids), len(val_battery_ids), len(test_battery_ids)]
seq_counts = [train_data['X'].shape[0], val_data['X'].shape[0], test_data['X'].shape[0]]

colors = ['#2980b9', '#27ae60', '#e67e22']

bars1 = ax1.bar(splits, bat_counts, color=colors, edgecolor='black', alpha=0.85)
ax1.set_title(f'Battery Count per Split ({total_bats} Total)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Batteries', fontsize=11)
ax1.grid(axis='y', alpha=0.3)
for b in bars1:
    y = b.get_height()
    ax1.text(b.get_x() + b.get_width()/2.0, y + 0.5, f"{int(y)} ({y/total_bats*100:.1f}%)", ha='center', va='bottom', fontsize=9, fontweight='bold')

bars2 = ax2.bar(splits, seq_counts, color=colors, edgecolor='black', alpha=0.85)
ax2.set_title(f'PyTorch LSTM Sequences per Split (W = {SEQUENCE_LENGTH})', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Sequences', fontsize=11)
ax2.grid(axis='y', alpha=0.3)
tot_s = sum(seq_counts)
for b in bars2:
    y = b.get_height()
    ax2.text(b.get_x() + b.get_width()/2.0, y + 25, f"{int(y):,} ({y/tot_s*100:.1f}%)", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
p1 = os.path.join(VIZ_DIR, 'split_distribution.png')
plt.savefig(p1, dpi=120, bbox_inches='tight')
plt.close()
print(f"Saved: {p1}")


Saved: data/intermediate/data/intermediate/lstm_prepared_data\visualizations\split_distribution.png


In [15]:
# ── Plot 2: Sample Target Trajectories Across Sequences (Train Split) ─────────
plt.figure(figsize=(12, 5))
train_meta = train_data['metadata']

sample_train_bats = ['Battery_01', 'Battery_02', 'Battery_04', 'Battery_24']
colors_bat = plt.cm.tab10(np.linspace(0, 1, len(sample_train_bats)))

for b_id, c in zip(sample_train_bats, colors_bat):
    b_meta = train_meta[train_meta['Battery_ID'] == b_id]
    plt.plot(b_meta['End_Cycle'], b_meta['SOH_Target'], label=f"{b_id} SOH", color=c, linewidth=1.5)
    
plt.axhline(80.0, color='red', linestyle='--', linewidth=1.2, label='80% EOL Threshold')
plt.title(f'SOH Targets Associated with Consecutive Sequence End-Cycles (W = {SEQUENCE_LENGTH})', fontsize=13, fontweight='bold')
plt.xlabel('Prediction Cycle (Sequence End Point)', fontsize=11)
plt.ylabel('Target SOH (%)', fontsize=11)
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()

p2 = os.path.join(VIZ_DIR, 'sequence_target_trajectories.png')
plt.savefig(p2, dpi=120, bbox_inches='tight')
plt.close()
print(f"Saved: {p2}")


Saved: data/intermediate/data/intermediate/lstm_prepared_data\visualizations\sequence_target_trajectories.png


### 8. Saving Sequential Datasets & Metadata
- Serializes PyTorch tensor files (`X_train.pt`, `y_train.pt`, etc.) and metadata.


In [17]:
# 1. Save Split Battery Assignments
split_records = []
for b in train_battery_ids:
    split_records.append({'Battery_ID': b, 'Split': 'Train', 'Total_Cycles': len(df[df['Battery_ID']==b])})
for b in val_battery_ids:
    split_records.append({'Battery_ID': b, 'Split': 'Validation', 'Total_Cycles': len(df[df['Battery_ID']==b])})
for b in test_battery_ids:
    split_records.append({'Battery_ID': b, 'Split': 'Test', 'Total_Cycles': len(df[df['Battery_ID']==b])})

split_df = pd.DataFrame(split_records)
split_csv_path = os.path.join(OUTPUT_DIR, 'split_battery_assignments.csv')
split_df.to_csv(split_csv_path, index=False)
print(f"Saved split assignments to: {split_csv_path}")

# 2. Save Feature Columns Info
feat_audit_df.to_csv(os.path.join(OUTPUT_DIR, 'feature_columns_info.csv'), index=False)
print(f"Saved feature info to: {OUTPUT_DIR}/feature_columns_info.csv")

# 3. Save Sequence Metadata CSVs
train_data['metadata'].to_csv(os.path.join(OUTPUT_DIR, 'train_sequences_metadata.csv'), index=False)
val_data['metadata'].to_csv(os.path.join(OUTPUT_DIR, 'val_sequences_metadata.csv'), index=False)
test_data['metadata'].to_csv(os.path.join(OUTPUT_DIR, 'test_sequences_metadata.csv'), index=False)
print("Saved sequence metadata CSVs.")

# 4. Save PyTorch Tensor Packages
torch.save({
    'X'       : train_data['X'],
    'y_soh'   : train_data['y_soh'],
    'y_rul'   : train_data['y_rul'],
    'mask_soh': train_data['mask_soh'],
    'mask_rul': train_data['mask_rul']
}, os.path.join(OUTPUT_DIR, 'train_tensors.pt'))

torch.save({
    'X'       : val_data['X'],
    'y_soh'   : val_data['y_soh'],
    'y_rul'   : val_data['y_rul'],
    'mask_soh': val_data['mask_soh'],
    'mask_rul': val_data['mask_rul']
}, os.path.join(OUTPUT_DIR, 'val_tensors.pt'))

torch.save({
    'X'       : test_data['X'],
    'y_soh'   : test_data['y_soh'],
    'y_rul'   : test_data['y_rul'],
    'mask_soh': test_data['mask_soh'],
    'mask_rul': test_data['mask_rul']
}, os.path.join(OUTPUT_DIR, 'test_tensors.pt'))
print("Saved PyTorch .pt tensor files (train_tensors.pt, val_tensors.pt, test_tensors.pt).")

# 5. Save Master JSON Config
config_data = {
    'sequence_length'     : SEQUENCE_LENGTH,
    'number_of_features'  : len(DISCHARGE_FEATURES),
    'feature_names'       : DISCHARGE_FEATURES,
    'target_names'        : TARGET_COLUMNS,
    'random_seed'         : 42,
    'train_batteries'     : train_battery_ids,
    'validation_batteries': val_battery_ids,
    'test_batteries'      : test_battery_ids,
    'train_shape'         : list(train_data['X'].shape),
    'val_shape'           : list(val_data['X'].shape),
    'test_shape'          : list(test_data['X'].shape)
}

config_path = os.path.join(OUTPUT_DIR, 'preparation_config.json')
with open(config_path, 'w') as f:
    json.dump(config_data, f, indent=2)
print(f"Saved master preparation configuration to: {config_path}")


Saved split assignments to: data/intermediate/data/intermediate/lstm_prepared_data\split_battery_assignments.csv
Saved feature info to: data/intermediate/data/intermediate/lstm_prepared_data/feature_columns_info.csv
Saved sequence metadata CSVs.
Saved PyTorch .pt tensor files (train_tensors.pt, val_tensors.pt, test_tensors.pt).
Saved master preparation configuration to: data/intermediate/data/intermediate/lstm_prepared_data\preparation_config.json


### 9. Sequential Engineering Summary
- Verifies ready tensors for multi-output LSTM training.


In [19]:
print("=" * 75)
print("NOTEBOOK 06: LSTM DATA PREPARATION COMPLETED SUCCESSFULLY")
print("=" * 75)
print(f"Prepared Outputs Directory : {os.path.abspath(OUTPUT_DIR)}")
print(f"Input Tensors Saved (PyTorch): train_tensors.pt, val_tensors.pt, test_tensors.pt")
print(f"Configuration File         : {os.path.abspath(config_path)}")
print(f"Ready for PyTorch Modeling : YES (Multi-Output LSTM Architecture)")
print("=" * 75)


NOTEBOOK 06: LSTM DATA PREPARATION COMPLETED SUCCESSFULLY
Prepared Outputs Directory : D:\deeplearning\project ml\data/intermediate/data/intermediate/lstm_prepared_data
Input Tensors Saved (PyTorch): train_tensors.pt, val_tensors.pt, test_tensors.pt
Configuration File         : D:\deeplearning\project ml\data/intermediate/data/intermediate/lstm_prepared_data\preparation_config.json
Ready for PyTorch Modeling : YES (Multi-Output LSTM Architecture)
